In [ ]:
import os
import re
import pandas as pd
import numpy as np

# ---- helpers ----
def normalize_columns(cols):
    # strip leading/trailing whitespace; collapse internal whitespace; keep original case if you want
    out = []
    for c in cols:
        c = str(c)
        c = c.strip()                  # removes trailing spaces like "ID_on_field "
        c = re.sub(r"\s+", " ", c)     # collapse multiple spaces
        out.append(c)
    return out

def merge_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    If a df has duplicate column names, merge them into one by taking first non-null across duplicates.
    """
    if df.columns.is_unique:
        return df

    # work with positions to drop duplicates safely
    cols = pd.Index(df.columns)
    dup_names = cols[cols.duplicated()].unique()

    for name in dup_names:
        block = df.loc[:, name]  # DataFrame with all columns called 'name'
        # take first non-null across duplicates
        df[name] = block.bfill(axis=1).iloc[:, 0]

        # drop extra duplicates (keep the first occurrence)
        idxs = [i for i, c in enumerate(df.columns) if c == name]
        for j in reversed(idxs[1:]):
            df.drop(columns=df.columns[j], inplace=True)

    return df

# ---- your metadata table ----
metadata = pd.DataFrame({
    "FILE": [
        "metadata_washed_2020.csv",
        "metadata_crushed_2021.csv",
        "metadata_crushed_2022.csv",
        "metadata_shotgun_2022_amplicon.csv",
        "metadata_washed_2022.csv",
        "metadata_shotgun_2022.csv",
        "metadata_crushed_2023_ITS.csv",
        "metadata_shotgun_2023.csv",
        "metadata_crushed_2023_16S.csv",
        "metadata_shotgun_2023_amplicon.csv",
        "metadata_crushed_seq_controls.csv"
    ],
    "LEAF_PROTOCOL": [
        "washed", "crushed", "crushed", "washed", "washed",
        "washed", "crushed", "washed", "crushed", "washed", "crushed"
    ],
    "LIBRARY_PREPARATION": [
        "amplicon", "amplicon", "amplicon", "shotgun_amplicon", "amplicon",
        "shotgun", "amplicon", "shotgun", "amplicon", "shotgun_amplicon", "amplicon"
    ],
    "YEAR": [
        "2020", "2021", "2022", "2022", "2022",
        "2022", "2023", "2023", "2023", "2023", "control"
    ]
})

data_directory = "."
dataframes = []

for _, row in metadata.iterrows():
    file_path = os.path.join(data_directory, row["FILE"])
    if not os.path.exists(file_path):
        print(f"File not found: {row['FILE']}")
        continue

    df = pd.read_csv(file_path, low_memory=False)

    # ✅ normalize column names immediately (fixes "ID_on_field " vs "ID_on_field")
    df.columns = normalize_columns(df.columns)

    # ✅ if any duplicates still exist within a single file, merge them
    df = merge_duplicate_columns(df)

    # add your metadata columns
    df = df.sort_index()
    df["LEAF_PROTOCOL"] = row["LEAF_PROTOCOL"]
    df["LIBRARY_PREPARATION"] = row["LIBRARY_PREPARATION"]
    df["YEAR"] = row["YEAR"]

    dataframes.append(df)

final_df = pd.concat(dataframes, ignore_index=True, sort=False)


# ✅ after concat, normalize again and merge again (catches duplicates created across files)
final_df.columns = final_df.columns.str.strip()

final_df.columns = normalize_columns(final_df.columns)
final_df = merge_duplicate_columns(final_df)

if "suggested_correct_seq_sample_ID" in final_df.columns:
    final_df = final_df.rename(columns={"suggested_correct_seq_sample_ID": "technical_sample_ID"})

LOC_MAP = {
    "R": "Ringsted",
    "H": "Holeby",
    "A": "Randers",
    "T": "Taastrup",
    "L": "Lokken",
    "S": "Sonderborg",
}

if "technical_sample_ID" in final_df.columns:

    s = final_df["technical_sample_ID"].astype(str).str.strip()

    # Match: R-xxxx, R_xxxx, R xxxx, etc.
    final_df["location_initial"] = s.str.extract(
        r"^\s*([RHALTS])\s*[-_./\s]+",
        expand=False
    )

    # Map to full location
    final_df["location"] = final_df["location_initial"].map(LOC_MAP)

else:
    print("technical_sample_ID column not found.")

# ---------------------------
# Optional: sanity check
# ---------------------------
print("Location counts:")
print(final_df["location"].value_counts(dropna=False))


# optional sanity check
dups = final_df.columns[final_df.columns.duplicated()].tolist()


print("Remaining duplicated columns:", dups)


final_df.to_csv("metagenomics_metadata_MATRIX.csv", index=False)
final_df

✅ Wrote location_year.csv
Rows: 8


,location_year_id,location,location_initial,year
0,LY__holeby__2022,Holeby,H,2022
1,LY__løkken__2022,Løkken,L,2022
2,LY__randers__2022,Randers,A,2022
3,LY__ringsted__2022,Ringsted,R,2022
4,LY__sønderborg__2022,Sønderborg,S,2022
5,LY__taastrup__2020,Taastrup,T,2020
6,LY__taastrup__2021,Taastrup,T,2021
7,LY__taastrup__2023,Taastrup,T,2023


In [ ]:
final_df[final_df["location"].isna()].to_csv("missing_location.csv", index=False) # check missing location info

In [ ]:
import pandas as pd
import numpy as np

# =========================
# CANONICAL (for location_year)
# =========================

canonical = pd.DataFrame(index=final_df.index)

# Clean year as string (and handle missing)
canonical["year"] = final_df["YEAR"].astype(str).str.strip() if "YEAR" in final_df.columns else np.nan

# location already derived from technical_sample_ID in your block
canonical["location"] = final_df["location"] if "location" in final_df.columns else np.nan
canonical["location_initial"] = final_df["location_initial"] if "location_initial" in final_df.columns else np.nan

# Optional: keep these for traceability/debugging (won't be required in the object table)
if "vila_ID" in final_df.columns:
    canonical["vila_ID"] = final_df["vila_ID"]

# Remove control rows for location_year
canonical = canonical[canonical["year"].str.lower().ne("control")].copy()

# Only keep rows where location exists (otherwise you can't form a location_year)
canonical = canonical[canonical["location"].notna()].copy()

# Create stable ID
canonical["location_year_id"] = (
    "LY__"
    + canonical["location"].astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False)
    + "__"
    + canonical["year"].astype(str).str.strip()
)

# =========================
# LOCATION_YEAR OBJECT TABLE
# =========================

location_year = (
    canonical[["location_year_id", "location", "location_initial", "year"]]
    .drop_duplicates()
    .sort_values(["location", "year"])
    .reset_index(drop=True)
)

location_year.to_csv("location_year.csv", index=False)

print("✅ Wrote location_year.csv")
print("Rows:", len(location_year))
location_year.head(25)

AttributeError: 'DataFrame' object has no attribute 'str'

In [13]:
df.columns.to_list()	

['bio_sample_ID',
 'vila_ID',
 'sample_number',
 'sample_date',
 'sample_type',
 'grid_number',
 'grid_point',
 'distance',
 'scheme',
 'sampling_0',
 'sampling_1',
 'edge',
 'block',
 'plot',
 'subplot',
 'ID_on_field',
 'spray_fungicide',
 'fertiliser',
 'cultivar',
 'density',
 'harvest_date',
 'yield_grain',
 'yield_straw',
 'yield_TGW',
 'sample_weight_plus_minus_2mg',
 'lyophilised_date',
 'crushed_date',
 'DNA_plate_number',
 'DNA_ext_date',
 'DNA_conc_plate',
 'lib_prep',
 'amplicon',
 'lib_plate_number',
 'plate_position',
 'PCR1_date',
 'PCR2_date',
 'lib_DNA_conc',
 'index_set',
 'S5_index_ID',
 'S5_index',
 'N7_index_ID',
 'N7_index',
 'index_pair',
 'seq_sample_ID',
 'run_ID',
 'note',
 'dir_name',
 'full_path',
 'forward_fq',
 'reverse_fq',
 'corrected_forward_fq',
 'corrected_reverse_fq',
 'current_seq_sample_ID',
 'correct_seq_sample_ID',
 'suggested_correct_seq_sample_ID',
 'SUGGESTED_EQUALS_CORRECT',
 'ALREADY_CORRECT',
 'LEAF_PROTOCOL',
 'LIBRARY_PREPARATION',
 'YEAR

In [4]:
# Function to convert stringified lists back into actual lists
def convert_to_list(val):
    try:
        # Use ast.literal_eval to safely convert stringified list to actual list
        return ast.literal_eval(val) if isinstance(val, str) else val
    except (ValueError, SyntaxError):
        return val  # Return the value as it is if it cannot be evaluated

# Apply the conversion to all columns
for column in ['forward_fq', 'reverse_fq', 'corrected_forward_fq', 'corrected_reverse_fq']:
    final_df[column] = final_df[column].apply(convert_to_list)

# Function to split rows for list-containing columns
def split_lists_in_specified_columns(df, columns_to_check):
    rows = []
    
    for _, row in df.iterrows():
        list_lengths = [len(row[column]) if isinstance(row[column], list) else 1 for column in columns_to_check]
        max_len = max(list_lengths)
        
        for i in range(max_len):
            new_row = row.copy()
            
            for j, column in enumerate(columns_to_check):
                if isinstance(new_row[column], list):
                    new_row[column] = new_row[column][i] if len(new_row[column]) > i else None
            
            rows.append(new_row)
    
    new_df = pd.DataFrame(rows)
    return new_df

# List of columns to check for lists
columns_to_split = ['forward_fq', 'reverse_fq', 'corrected_forward_fq', 'corrected_reverse_fq']

# Apply the function to the specified columns
final_df_long=split_lists_in_specified_columns(final_df, columns_to_split)
# # Display the resulting DataFrame
# import ace_tools as tools; tools.display_dataframe_to_user(name="Split List Columns", dataframe=final_df)
final_df_long

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,grid_number,grid_point,distance,scheme,sampling_0,...,DNA_conc_individual,letter,Days_between_DNA_x,run_number,speedvac_DNA_conc_individual,lib_date,x,y,number_bio,number_sample
0,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T-030620-w-24,NaN,24.0,2020-06-03,deep,26.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T-030620-w-23,NaN,23.0,2020-06-03,deep,25.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,T-030620-w-22,NaN,22.0,2020-06-03,deep,24.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T-030620-w-21,NaN,21.0,2020-06-03,deep,23.0,middle,cross,large,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5841,A-230622-c-51-5,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-5,51-5
5842,A-230622-c-51-4,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-4,51-4
5843,A-230622-c-51-3,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-3,51-3
5844,A-230622-c-57,NaN,57.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57,57


In [5]:
# Create the first modified DataFrame
df1 = final_df_long.copy()
df1 = df1.drop(columns=['forward_fq', 'corrected_forward_fq'])
df1 = df1.rename(columns={'reverse_fq': 'fq_name', 'corrected_reverse_fq': 'corrected_fq_name'})

# Create the second modified DataFrame
df2 = final_df_long.copy()
df2 = df2.drop(columns=['reverse_fq', 'corrected_reverse_fq'])
df2 = df2.rename(columns={'forward_fq': 'fq_name', 'corrected_forward_fq': 'corrected_fq_name'})

# Combine the two DataFrames by stacking them on top of each other
combined_df = pd.concat([df1, df2], ignore_index=True)
print(len(combined_df))
# NOTE there are some empty rows
combined_na_df=combined_df[combined_df["fq_name"].isna()]#.sort_values("fq_name")["fq_name"][-5:].to_list()
combined_df=combined_df[~combined_df["fq_name"].isna()]#.sort_values("fq_name")["fq_name"][-5:].to_list()
combined_df

11692


,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,grid_number,grid_point,distance,scheme,sampling_0,...,DNA_conc_individual,letter,Days_between_DNA_x,run_number,speedvac_DNA_conc_individual,lib_date,x,y,number_bio,number_sample
0,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T-030620-w-24,NaN,24.0,2020-06-03,deep,26.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T-030620-w-23,NaN,23.0,2020-06-03,deep,25.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,T-030620-w-22,NaN,22.0,2020-06-03,deep,24.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T-030620-w-21,NaN,21.0,2020-06-03,deep,23.0,middle,cross,large,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11687,A-230622-c-51-5,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-5,51-5
11688,A-230622-c-51-4,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-4,51-4
11689,A-230622-c-51-3,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-3,51-3
11690,A-230622-c-57,NaN,57.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57,57


In [6]:
combined_na_df

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,grid_number,grid_point,distance,scheme,sampling_0,...,DNA_conc_individual,letter,Days_between_DNA_x,run_number,speedvac_DNA_conc_individual,lib_date,x,y,number_bio,number_sample
108,T-150621-c-mystery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
406,T-150621-c-128,NaN,128.0,2021-06-15,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
407,T-150621-c-127,NaN,127.0,2021-06-15,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
408,T-150621-c-126,NaN,126.0,2021-06-15,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
409,T-150621-c-125,NaN,125.0,2021-06-15,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11341,A-230622-c-60-2,NaN,60.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60-2,61-2
11452,A-230622-c-69,NaN,69.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69,69
11455,A-230622-c-71-1,NaN,71.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71-1,71-1
11456,A-230622-c-71-2,NaN,71.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71-2,71-2


In [7]:
combined_df['fq_name'] = combined_df.apply(
    lambda row: row['fq_name'] if '/' in row['fq_name'] else row['full_path'] + "/" + row['fq_name'],
    axis=1
)
combined_df['corrected_fq_name'] = combined_df.apply(
    lambda row: row['corrected_fq_name'] if '/' in row['corrected_fq_name'] else row['full_path'] + "/" + row['corrected_fq_name'],
    axis=1
)
combined_df#.sort_values("fq_name")#["fq_name"][-5:].to_list()



,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,grid_number,grid_point,distance,scheme,sampling_0,...,DNA_conc_individual,letter,Days_between_DNA_x,run_number,speedvac_DNA_conc_individual,lib_date,x,y,number_bio,number_sample
0,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T-030620-w-24,NaN,24.0,2020-06-03,deep,26.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T-030620-w-23,NaN,23.0,2020-06-03,deep,25.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,T-030620-w-22,NaN,22.0,2020-06-03,deep,24.0,middle,other,medium,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,T-030620-w-21,NaN,21.0,2020-06-03,deep,23.0,middle,cross,large,C_10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11687,A-230622-c-51-5,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-5,51-5
11688,A-230622-c-51-4,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-4,51-4
11689,A-230622-c-51-3,NaN,51.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51-3,51-3
11690,A-230622-c-57,NaN,57.0,2022-06-23,deep,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57,57


In [8]:
repeated_df=combined_df.value_counts(["fq_name"]).sort_values().to_frame()
repeated_df=repeated_df[repeated_df["count"]>1]
repeated=repeated_df.reset_index()["fq_name"].to_list()
repeated_df
# 15284

,count
fq_name,
/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230707_DWDQH/A-230622-16S-c-59-3_S126_L001_R2_001.fastq.gz,2
/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230707_DWDQH/A-230622-16S-c-58-3_S119_L001_R2_001.fastq.gz,2
/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230707_DWDQH/A-230622-16S-c-58-4_S120_L001_R2_001.fastq.gz,2
/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230707_DWDQH/A-230622-16S-c-59-5_S128_L001_R2_001.fastq.gz,2
/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230707_DWDQH/A-230622-16S-c-59-6_S129_L001_R2_001.fastq.gz,2
...,...
/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230526_JRV78/A-230622-ITS-c-63-3_S153_L001_R1_001.fastq.gz,2
/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230526_JRV78/A-230622-ITS-c-63-4_S154_L001_R1_001.fastq.gz,2
/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230526_JRV78/A-230622-ITS-c-63-5_S155_L001_R1_001.fastq.gz,2


In [9]:
test_string="/home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq230526_JRV78A-230622-ITS-c-70-5_S168_L001_R1_001.fastq.gz"
combined_df[combined_df["fq_name"].isin(repeated)].to_csv("error.tsv")
# combined_df[˜combined_df["fq_name"].isin(repeated)]


In [10]:
read_list=pd.read_csv("../trinucleotide_count_full.csv", sep="\t")[["Path"]]
read_list[ "folder"]=read_list[ "Path"].str.split("/", n=1).str[0]
read_list=read_list[~read_list["Path"].str.contains("error")].reset_index(drop=True)
read_list.merge(combined_df, left_on="Path", right_on="fq_name", how="outer").to_csv( "test_full_files.csv")
read_list

,Path,folder
0,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,
1,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,
2,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,
3,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,
4,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,
...,...,...
14770,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,
14771,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,
14772,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,
14773,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,


In [11]:
final_read_list=read_list.merge(combined_df, left_on="Path", right_on="fq_name", how="outer")
final_read_list["origin"]=final_read_list["Path"].str.split( "/", n=10).str[-1]
final_read_list["destination"]=final_read_list["corrected_fq_name"].str.split( "/", n=11).str[-1]
final_read_list.to_csv("full_files.csv")

In [12]:
final_read_list[~final_read_list["corrected_fq_name"].isna()]#.to_csv( "full_files_notna.csv", sep="\t")
final_read_list[~final_read_list["corrected_fq_name"].isna()][["origin", "destination"]]#.to_csv( "full_files_notna_names.csv", sep="\t", header=False, index=False)


,origin,destination
1826,seq230526_JRV78/A-230622-ITS-c-49-1_S90_L001_R...,A-230622-ITS-c-49-1_S90_L001_R1_001.fastq.gz
1827,seq230526_JRV78/A-230622-ITS-c-49-1_S90_L001_R...,A-230622-ITS-c-49-1_S90_L001_R2_001.fastq.gz
1828,seq230526_JRV78/A-230622-ITS-c-49-2_S91_L001_R...,A-230622-ITS-c-49-2_S91_L001_R1_001.fastq.gz
1829,seq230526_JRV78/A-230622-ITS-c-49-2_S91_L001_R...,A-230622-ITS-c-49-2_S91_L001_R2_001.fastq.gz
1830,seq230526_JRV78/A-230622-ITS-c-49-3_S92_L001_R...,A-230622-ITS-c-49-3_S92_L001_R1_001.fastq.gz
...,...,...
15273,seq240826_X6DU5/T-300523-ITS-c-91A-2_S21_L001_...,T-300523-ITS-c-91A-2_S21_L001_R2_001.fastq.gz
15274,seq240826_X6DU5/T-300523-ITS-c-98A-2_S22_L001_...,T-300523-ITS-c-98A-2_S22_L001_R1_001.fastq.gz
15275,seq240826_X6DU5/T-300523-ITS-c-98A-2_S22_L001_...,T-300523-ITS-c-98A-2_S22_L001_R2_001.fastq.gz
15276,seq240826_X6DU5/T-300523-ITS-c-9A-2_S5_L001_R1...,T-300523-ITS-c-9A-2_S5_L001_R1_001.fastq.gz


In [13]:
final_read_list[final_read_list["corrected_fq_name"].isna()]

,Path,folder,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,grid_number,grid_point,distance,...,Days_between_DNA_x,run_number,speedvac_DNA_conc_individual,lib_date,x,y,number_bio,number_sample,origin,destination
0,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq221122_ISEQ1/MATRIX_meta_test_4A_shotgun_S4...,NaN
1,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq221122_ISEQ1/MATRIX_meta_test_4A_shotgun_S4...,NaN
2,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq221122_ISEQ1/MATRIX_meta_test_4E_shotgun_S4...,NaN
3,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq221122_ISEQ1/MATRIX_meta_test_4E_shotgun_S4...,NaN
4,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq221122_ISEQ1/MATRIX_meta_test_4G_shotgun_S4...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14303,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq240820_C6WGP/T-130623-ITS-c-10C-1_S49_L001_...,NaN
14318,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq240820_C6WGP/T-180723-ITS-c-26G-1_S50_L001_...,NaN
14319,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq240820_C6WGP/T-180723-ITS-c-26G-1_S50_L001_...,NaN
14372,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,seq240820_C6WGP/T-180723-ITS-s-35H_S51_L001_R1...,NaN
